In [9]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.pytorch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

print(f"MLflow versão: {mlflow.__version__}")

MLflow versão: 3.11.1


## MLOps com MLflow

**O que é MLOps?**
MLOps (Machine Learning Operations) é a prática de gerenciar modelos 
de ML em produção de forma organizada e reproduzível.

**O problema sem MLOps:**
- Você treina 10 modelos diferentes no notebook
- Semanas depois não lembra qual parâmetro usou no melhor modelo
- Não consegue reproduzir os resultados
- Não sabe quando o modelo começou a degradar

**O que o MLflow resolve:**
- Registra automaticamente: parâmetros, métricas e artefatos de cada experimento
- Compara visualmente diferentes runs
- Versiona os modelos (v1, v2, v3...)
- Permite reproduzir qualquer experimento com um clique

É como o Git, mas para modelos de ML.

In [10]:
import os
os.chdir(r'C:\Users\joaop\projeto-churn-mlops')

df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop(['customerID', 'gender', 'PhoneService'], axis=1)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
categoricas = df.select_dtypes(include='object').columns.tolist()
df_ml = pd.get_dummies(df, columns=categoricas, drop_first=True)

X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

mlflow.set_tracking_uri('file:mlruns')
mlflow.set_experiment('churn-prediction')

print("Dados preparados!")
print("Experimento MLflow configurado: churn-prediction")

Dados preparados!
Experimento MLflow configurado: churn-prediction


### Registrando experimentos no MLflow

Cada `mlflow.start_run()` cria um "run" — um registro completo de:
- Parâmetros usados (n_estimators, learning_rate, etc.)
- Métricas obtidas (acurácia, AUC-ROC, F1)
- Artefatos (o modelo salvo)

Vamos rodar 3 modelos e registrar tudo automaticamente.

In [11]:
# Registrar 3 experimentos no MLflow
modelos_config = [
    {
        'nome': 'Logistic Regression',
        'modelo': LogisticRegression(max_iter=1000, random_state=42),
        'params': {'max_iter': 1000, 'random_state': 42}
    },
    {
        'nome': 'Random Forest',
        'modelo': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
        'params': {'n_estimators': 200, 'max_depth': 10, 'random_state': 42}
    },
    {
        'nome': 'Gradient Boosting Otimizado',
        'modelo': GradientBoostingClassifier(
            learning_rate=0.05, max_depth=7, 
            min_samples_split=5, n_estimators=300, random_state=42
        ),
        'params': {'learning_rate': 0.05, 'max_depth': 7, 
                   'min_samples_split': 5, 'n_estimators': 300}
    }
]

for config in modelos_config:
    with mlflow.start_run(run_name=config['nome']):
        
        # Logar parâmetros
        mlflow.log_params(config['params'])
        
        # Treinar
        config['modelo'].fit(X_train_scaled, y_train_bal)
        
        # Prever
        y_pred = config['modelo'].predict(X_test_scaled)
        y_proba = config['modelo'].predict_proba(X_test_scaled)[:, 1]
        
        # Calcular métricas
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        f1 = f1_score(y_test, y_pred)
        
        # Logar métricas
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('auc_roc', auc)
        mlflow.log_metric('f1_score', f1)
        
        # Logar modelo
        mlflow.sklearn.log_model(config['modelo'], 'model')
        
        print(f"✓ {config['nome']}")
        print(f"  Acurácia: {acc*100:.1f}% | AUC: {auc:.3f} | F1: {f1:.3f}")
        print(f"  Run registrado no MLflow\n")

2026/05/03 10:58:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 10:58:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Logistic Regression
  Acurácia: 74.8% | AUC: 0.808 | F1: 0.572
  Run registrado no MLflow



2026/05/03 10:59:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 10:59:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Random Forest
  Acurácia: 76.7% | AUC: 0.836 | F1: 0.606
  Run registrado no MLflow



2026/05/03 11:00:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 11:00:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Gradient Boosting Otimizado
  Acurácia: 77.7% | AUC: 0.819 | F1: 0.588
  Run registrado no MLflow



### O que o MLflow registrou pra cada modelo:

**Parâmetros:** os hiperparâmetros usados em cada treino
(n_estimators, learning_rate, max_depth, etc.)

**Métricas:** acurácia, AUC-ROC e F1-score

**Artefatos:** o modelo serializado (arquivo .pkl) pronto 
pra ser carregado e usado em produção

**O valor pra empresa:**
- Qualquer pessoa do time consegue reproduzir qualquer experimento
- Fica claro qual modelo é o melhor e por quê
- Quando o modelo degradar em produção, você sabe exatamente 
  com quais dados e parâmetros ele foi treinado
- Auditoría completa de todos os experimentos realizados